# AI Predictive Maintenance & Simulation Platform
An educational end-to-end project: synthetic industrial telemetry, failure-risk prediction, maintenance recommendations, and what-if policy simulation.

**Safety:** Synthetic data only. This prototype is not validated for real machinery or safety-critical decisions. A qualified engineer must review all real-world actions.

## 1. Install and import dependencies

In [ ]:
# In a fresh environment, run: %pip install numpy pandas matplotlib scikit-learn
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

SEED=42
rng=np.random.default_rng(SEED)
OUT=Path("outputs"); OUT.mkdir(exist_ok=True)

## 2. Generate synthetic equipment telemetry

In [ ]:
def generate_data(n_assets=24, days=180, seed=42):
    r=np.random.default_rng(seed); rows=[]
    for i in range(n_assets):
        asset=f"ASSET-{i+1:03d}"
        kind=["pump","motor","compressor"][i%3]
        age=r.uniform(1,12); vib0=r.uniform(1,2.5); temp0=r.uniform(55,72)
        degradation=r.uniform(.002,.012); health=1.0
        for d in range(days):
            health=max(.05, health-degradation*r.uniform(.2,1.8))
            vib=vib0+(1-health)*5+r.normal(0,.25)
            temp=temp0+(1-health)*22+r.normal(0,2)
            pressure=100+(1-health)*18+r.normal(0,3)
            current=10+(1-health)*4+r.normal(0,.8)
            hours=r.uniform(14,24)
            risk=.30*max(0,vib-3)/5+.25*max(0,temp-78)/25+.20*max(0,pressure-110)/25+.15*max(0,current-12)/5+.10*(1-health)
            p=1/(1+np.exp(-7*(risk-.45)))
            rows.append([pd.Timestamp("2025-01-01")+pd.Timedelta(days=d),asset,kind,age,hours,max(0,vib),temp,pressure,max(0,current),int(r.random()<p)])
    return pd.DataFrame(rows,columns=["timestamp","asset_id","asset_type","age_years","operating_hours","vibration_mm_s","temperature_c","pressure_kpa","motor_current_a","failure_within_7d"])

data=generate_data()
display(data.head())
print("Rows:",len(data)," | Assets:",data.asset_id.nunique()," | Positive rate:",round(data.failure_within_7d.mean(),3))

## 3. Explore data and engineer rolling features

In [ ]:
sensor_cols=["vibration_mm_s","temperature_c","pressure_kpa","motor_current_a"]
display(data[sensor_cols].describe().T)
data=data.sort_values(["asset_id","timestamp"]).copy()
for col in sensor_cols:
    data[f"{col}_mean7"]=data.groupby("asset_id")[col].transform(lambda s:s.rolling(7,min_periods=1).mean())
    data[f"{col}_std7"]=data.groupby("asset_id")[col].transform(lambda s:s.rolling(7,min_periods=2).std())
data=data.fillna(0)
features=["age_years","operating_hours",*sensor_cols,*[f"{c}_mean7" for c in sensor_cols],*[f"{c}_std7" for c in sensor_cols]]

## 4. Train and evaluate a failure-risk model

In [ ]:
X=data[features]; y=data.failure_within_7d
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
model=Pipeline([("imputer",SimpleImputer(strategy="median")),
                ("rf",RandomForestClassifier(n_estimators=220,max_depth=10,min_samples_leaf=3,class_weight="balanced",random_state=SEED,n_jobs=-1))])
model.fit(X_train,y_train)
pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
print(classification_report(y_test,pred,zero_division=0))
print("ROC-AUC:",round(roc_auc_score(y_test,prob),3))
display(pd.DataFrame(confusion_matrix(y_test,pred),index=["Actual 0","Actual 1"],columns=["Predicted 0","Predicted 1"]))

## 5. Score equipment and recommend maintenance

In [ ]:
scored=data.copy()
scored["failure_risk_7d"]=model.predict_proba(scored[features])[:,1]
def action(p):
    if p>=.70: return ("Urgent inspection","High")
    if p>=.40: return ("Schedule preventive maintenance","Medium")
    return ("Continue monitoring","Low")
scored[["recommended_action","risk_band"]]=scored.failure_risk_7d.apply(lambda p:pd.Series(action(p)))
latest=scored.groupby("asset_id",as_index=False).tail(1).sort_values("failure_risk_7d",ascending=False)
display(latest[["asset_id","asset_type","failure_risk_7d","risk_band","recommended_action",*sensor_cols]].head(12))

## 6. Visualize risk and sensor trends

In [ ]:
plt.figure(figsize=(11,4))
plt.bar(latest.asset_id,latest.failure_risk_7d)
plt.axhline(.70,ls="--",label="Urgent threshold"); plt.axhline(.40,ls=":",label="Preventive threshold")
plt.xticks(rotation=75); plt.ylabel("Estimated probability"); plt.title("Latest 7-day failure risk by asset"); plt.legend(); plt.tight_layout(); plt.show()

asset=latest.iloc[0].asset_id
hist=scored[scored.asset_id==asset]
plt.figure(figsize=(10,4)); plt.plot(hist.timestamp,hist.vibration_mm_s,label="Vibration")
plt.title(f"Vibration trend — {asset}"); plt.ylabel("mm/s"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Simulate maintenance policies

In [ ]:
def simulate(frame, threshold=.40, inspection_cost=250, failure_cost=5000,
             planned_hours=2, unplanned_hours=18, seed=7):
    r=np.random.default_rng(seed)
    f=frame.copy(); f["risk"]=model.predict_proba(f[features])[:,1]
    base_fail=int(f.failure_within_7d.sum())
    base={"policy":"No preventive maintenance","failures":base_fail,"maintenance_actions":0,
          "downtime_hours":base_fail*unplanned_hours,"estimated_cost":base_fail*failure_cost}
    actions=failures=0; downtime=cost=0.0
    for row in f.itertuples():
        if row.risk>=threshold:
            actions+=1; cost+=inspection_cost; downtime+=planned_hours
            fail=bool(row.failure_within_7d) and (r.random()>.65) # illustrative intervention assumption
        else: fail=bool(row.failure_within_7d)
        if fail: failures+=1; cost+=failure_cost; downtime+=unplanned_hours
    policy={"policy":f"Risk threshold {threshold:.2f}","failures":failures,"maintenance_actions":actions,
            "downtime_hours":downtime,"estimated_cost":cost}
    return pd.DataFrame([base,policy])

simulation=simulate(scored)
display(simulation)

## 8. Compare simulated outcomes

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
simulation.plot(x="policy",y="estimated_cost",kind="bar",ax=axes[0],legend=False,title="Estimated cost")
simulation.plot(x="policy",y="downtime_hours",kind="bar",ax=axes[1],legend=False,title="Downtime hours")
for ax in axes: ax.set_xlabel(""); ax.tick_params(axis="x",rotation=15)
plt.tight_layout(); plt.show()
print("Illustrative cost difference:",simulation.iloc[0].estimated_cost-simulation.iloc[1].estimated_cost)
print("The intervention effect is assumed for demonstration, not estimated from causal evidence.")

## 9. Export recommendations and simulation report

In [ ]:
recommendations=latest[["asset_id","asset_type","timestamp","failure_risk_7d","risk_band","recommended_action",*sensor_cols]]
recommendations.to_csv(OUT/"asset_risk_recommendations.csv",index=False)
simulation.to_csv(OUT/"maintenance_policy_simulation.csv",index=False)
report={"project":"AI Predictive Maintenance & Simulation Platform",
        "assets":int(data.asset_id.nunique()),"rows":int(len(data)),
        "test_roc_auc":float(roc_auc_score(y_test,prob)),
        "simulation":simulation.to_dict(orient="records")}
(OUT/"maintenance_report.json").write_text(json.dumps(report,indent=2,default=str))
print("Exported:",[p.name for p in OUT.iterdir()])

## 10. Limitations and production roadmap

- All data and failure labels are synthetic; model scores are not real-world reliability estimates.
- Random train/test splitting is only a teaching baseline. Use time-based and asset-based validation for real deployment.
- The what-if simulator assumes maintenance suppresses some failures; this is a hand-set scenario, not a measured causal effect.
- Costs, thresholds, and sensor limits are illustrative and must be calibrated by reliability engineers.
- Future work: governed IoT/historian ingestion, remaining-useful-life models, probability calibration, SHAP explanations, drift monitoring, CMMS integration with human approval, and a Streamlit dashboard.
- Never connect this prototype directly to industrial control systems or automate safety-critical shutdowns.